# Reproducing Mountoufaris et al. (2024) from dandiset 001131: rSLDS checks, seed 0

Paper. Mountoufaris G, Nair A, et al. (2024), *Cell* 187, 5998-6015,
[doi:10.1016/j.cell.2024.08.015](https://doi.org/10.1016/j.cell.2024.08.015); the claims file was
written from the accepted author manuscript (PMC11490375). Dandiset
[DANDI:001131](https://dandiarchive.org/dandiset/001131/0.260601.1203), version 0.260601.1203 (see
`reproduce_mountoufaris_2024.ipynb` for the choice of version and the data checks).

The rSLDS claims are tested in `rslds_seeds_0_1.ipynb` and `rslds_seeds_2_3.ipynb`, on the z-scored
traces, with the settings of the claims file (fig6-rslds-fit-quality). This notebook runs, for seed
0 only, two checks the verifier of round 1 asked for:

1. The claims file's second reading of the trace, sigma-normalized (each unit's z-scored trace minus
   its mean over the assumed pre-intruder baseline, divided by its SD there; baseline `all`, the
   frames before the first annotated behaviour). Mouse 488, with 6.2 s before its first annotation,
   has no baseline and is not fit.
2. A held-out score: each mouse is fit with a fifth of its recording held out (4 of 20 blocks,
   spread over the session), and the emissions reconstruction and the 1-s and 5-s forward
   predictions of the activity are scored on the held-out frames (`vmh.fit_heldout`). The paper's
   score ("cvR2") uses held-out folds that the Methods do not describe; this is one fold.

Both use alpha = 0 and the other settings of the main fits. A check does not decide an outcome;
it shows whether these choices could change one.

| Claim | Here |
|---|---|
| fig6-rslds-fit-quality, fig6a, fig6c, fig6f, fig6g | checks only (sigma-normalized fits; held-out score) |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import vmh

print("ssm commit", vmh.ensure_ssm())
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore")

SEED = 0
started = time.time()
mice = vmh.load_all()
G = {m["mouse"]: m["group"] for m in mice}
M = {m["mouse"]: m for m in mice}
print(f"loaded {len(mice)} files from dandiset {vmh.DANDISET_ID} version {vmh.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
sig = {m["mouse"]: vmh.sigma_trace(m, "all") for m in mice}
rows = []
for m in mice:
    s = sig[m["mouse"]]
    rows.append({"mouse": m["mouse"], "group": m["group"], "frames": m["Y"].shape[0], "units": m["Y"].shape[1],
                 "baseline_s": None if s is None else len(vmh.baseline_frames(m, "all")) / m["rate"],
                 "sigma_min": None if s is None else np.nanmin(s), "sigma_max": None if s is None else np.nanmax(s),
                 "sigma_nan": None if s is None else int(np.isnan(s).sum())})
print(pd.DataFrame(rows).round(2).to_string(index=False))
sig = {k: v for k, v in sig.items() if v is not None}
print("mice with a baseline (sigma-normalized fits):", len(sig), "; seed", SEED, "; alpha 0")

## Sigma-normalized fits

In [ ]:
t0 = time.time()
smice = [m for m in mice if m["mouse"] in sig]
sfits = vmh.fit_all(smice, seeds=(SEED,), traces=sig)
print(f"{len(sfits)} fits in {(time.time() - t0) / 60:.1f} min")
rows = []
for (mouse, seed), f in sorted(sfits.items(), key=lambda kv: (G[kv[0][0]], kv[0][0])):
    s = vmh.summarize_fit(f, M[mouse])
    rows.append({"mouse": mouse, "group": G[mouse], "r2": s["r2"],
                 "elbo_change_last10_pct": 100 * (f["elbos"][-1] - f["elbos"][-11]) / abs(f["elbos"][-1]),
                 "fwd_r2_1s": vmh.forward_sim_score(f, 1.0), "fwd_r2_5s": vmh.forward_sim_score(f, 5.0),
                 "state_from": s["state_from"], "attack_in_state": s["attack_frac_in_state"],
                 **{f"tau{j + 1}_a": s["tau_a"][j] for j in range(vmh.D_LATENT)},
                 **{f"tau{j + 1}_b": s["tau_b"][j] for j in range(vmh.D_LATENT)},
                 "score_a": s["score_a"], "score_b": s["score_b"]})
sfit = pd.DataFrame(rows)
print(sfit.round(3).to_string(index=False))

In [ ]:
rows = []
for reading in ("a", "b"):
    out = {"reading": reading}
    for g in ("control", "experimental"):
        d = sfit[sfit.group == g]
        T = d[[f"tau{j + 1}_{reading}" for j in range(vmh.D_LATENT)]].to_numpy()
        out.update({f"r2_{g[:3]}": d.r2.mean(), f"tau1_{g[:3]}": T[:, 0].mean(), f"tau2_{g[:3]}": T[:, 1].mean(),
                    f"kw_p_{g[:3]}": stats.kruskal(*T.T).pvalue, f"dunn_1v2_p_{g[:3]}": vmh.dunn_1_vs_2(T),
                    f"score_{g[:3]}": d[f"score_{reading}"].mean(),
                    f"score_above_1_{g[:3]}": f"{int((d[f'score_{reading}'] > 1).sum())} of {len(d)}"})
    c, e = sfit[sfit.group == "control"], sfit[sfit.group == "experimental"]
    out["p_tau1_control_greater"] = vmh.mw(c[f"tau1_{reading}"], e[f"tau1_{reading}"], "greater")[1]
    out["p_score"] = vmh.mw(c[f"score_{reading}"], e[f"score_{reading}"])[1]
    rows.append(out)
print(pd.DataFrame(rows).round(3).T.to_string())
c, e = sfit[sfit.group == "control"], sfit[sfit.group == "experimental"]
att = sfit[sfit.state_from == "attack"]
print(f"\nvariance explained: control {c.r2.mean():.3f}, experimental {e.r2.mean():.3f} (range {sfit.r2.min():.3f} to {sfit.r2.max():.3f}); expected 0.85 +/- 0.085")
print(f"forward R2, Mann-Whitney p: 1 s {vmh.mw(c.fwd_r2_1s, e.fwd_r2_1s)[1]:.3f}, 5 s {vmh.mw(c.fwd_r2_5s, e.fwd_r2_5s)[1]:.3f}")
print(f"attack mostly in one state in {int((att.attack_in_state > 0.5).sum())} of {len(att)} mice, at least 90% in {int((att.attack_in_state >= 0.9).sum())}")
print("expected: control tau1 90 to 132 s, tau2 36 to 44 s; experimental tau1 below 55 s; control score 1.44 to 1.76")

## Held-out score (z-scored traces)

In [ ]:
from concurrent.futures import ProcessPoolExecutor

t0 = time.time()
jobs = sorted([(m["mouse"], m["Y"], SEED, vmh.NUM_ITERS) for m in mice], key=lambda j: -j[1].size)
with ProcessPoolExecutor(4) as ex:
    held = pd.DataFrame(list(ex.map(vmh.fit_heldout, jobs)))
held["group"] = held.mouse.map(G)
held = held.sort_values(["group", "mouse"])
print(f"{len(held)} held-out fits in {(time.time() - t0) / 60:.1f} min")
print(held.round(3).to_string(index=False))
c, e = held[held.group == "control"], held[held.group == "experimental"]
for col in ("test_r2", "test_fwd_r2_1s", "test_fwd_r2_5s"):
    print(f"{col}: control {c[col].mean():.3f}, experimental {e[col].mean():.3f}, "
          f"range {held[col].min():.3f} to {held[col].max():.3f}, Mann-Whitney p = {vmh.mw(c[col], e[col])[1]:.3f}")
print("expected variance explained 0.85 +/- 0.085; performance not different between groups at p < 0.05")

## Summary

Sigma-normalized fits (seed 0, the 9 mice with a baseline). Variance explained is 0.690 in control
and 0.636 in experimental mice (range 0.524 to 0.911), below 85 +/- 8.5% in both groups. Control
mean tau1 is 380 to 462 s and tau2 272 to 292 s, with Dunn's comparison of dimensions 1 and 2 at p =
1.0; experimental tau1 is 140 to 181 s, not below 55 s; the mean control line attractor score is
0.81 to 0.84, with 1 of 4 control mice above 1. Control tau1 is longer than experimental at p =
0.008 under reading (b), but the other conditions of fig6c-experimental-tau-indistinguishable fail.
Attack frames fall mostly in one state in 6 of the 7 mice with attack annotations, at least 90% in 1.
Unlike the z-scored fits, the in-sample forward R2 differs between groups (p = 0.016 at 1 s and 5 s).
The sigma-normalized reading would not change any rSLDS outcome: each still fails on the values the
claims file expects.

Held-out score (z-scored, seed 0, one fold of a fifth of each recording). The emissions
reconstruction explains 0.479 (control) and 0.410 (experimental) of the held-out variance (range
0.297 to 0.577), less than in-sample and further from 85%. Forward predictions of the held-out
activity do not differ between groups over 1 s (0.472 against 0.394, p = 0.26) but do over 5 s
(0.371 against 0.198, p = 0.010). So the paper's similar performance between groups holds here at
the 1-s horizon only; on the held-out score at 5 s and on the sigma-normalized fits (p = 0.016 at
both horizons) that part of the claim fails too. fig6-rslds-fit-quality fails on variance explained under every reading, so
this check does not change its outcome.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.